# VisDrone YOLOv8m Fine-tuning (Google Colab · GPU)

Mirrors `src/training/train.py`. Run on **any Colab GPU** — a T4 (free) or an A100 (Pro) both work; an A100 finishes in ~1 h. Same YOLOv8m config, hyperparameters, epochs, and DagsHub/MLflow logging as the script.

**Reference result:** this 50-epoch YOLOv8m fine-tune reached mAP@0.5 ≈ 0.435 on VisDrone-DET val in ~47 min on a Colab A100-80GB (VisDrone is a hard small-object aerial benchmark). Report whatever your run actually produces.

## Colab run steps
1. **Set the GPU:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU (or A100 / any GPU)*.
2. **Add your DagsHub token as a Colab secret:** open the 🔑 panel (left sidebar), create a secret named **`DAGSHUB_TOKEN`** with your DagsHub access token, and enable *Notebook access*.
3. **Mount Google Drive** (the cell below) so the raw dataset zips and the final `best.pt` survive runtime recycling.
4. Set your DagsHub MLflow URI + username in the **Config** cell.
5. *Runtime → Run all*.

**Storage model (fully automatic on Run All):** raw zips are cached on Drive, and the converted YOLO dataset is cached as a single tarball on Drive then extracted to fast local `/content` each session (avoids the Drive small-file FUSE hang and never re-converts once cached). Training checkpoints are written to Drive every 5 epochs, so a disconnect auto-resumes from `last.pt` on the next *Run All* — no manual steps. The final `best.pt` is copied to Drive at the end.

**You only ever need to: pick a GPU, confirm the `DAGSHUB_TOKEN` secret, and *Run All*.** No editing, zipping, copying, or path changes.

In [ ]:
# --- Install deps (Colab doesn't preinstall these) ---
!pip -q install ultralytics mlflow dagshub onnx onnxruntime

In [ ]:
# --- Verify GPU (T4, A100, or any CUDA GPU is fine) ---
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> GPU (T4/A100).'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# --- Mount Google Drive (persists dataset cache + checkpoints) ---
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/object-detection-tracking'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive project root:', DRIVE_ROOT)

In [ ]:
# --- Config: FILL THESE IN ---
import os

# DagsHub MLflow URI + username for this project (username is not secret).
MLFLOW_TRACKING_URI = 'https://dagshub.com/mohanemg07-web/object-detection-tracking.mlflow'
DAGSHUB_USERNAME = 'mohanemg07-web'

# Read the DagsHub token from Colab Secrets (never hardcode/commit it).
# os.environ is required here: a shell `!export` would NOT reach this kernel.
from google.colab import userdata
DAGSHUB_TOKEN = userdata.get('DAGSHUB_TOKEN')

os.environ['MLFLOW_TRACKING_URI'] = MLFLOW_TRACKING_URI
os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN or ''  # token -> MLflow password
assert DAGSHUB_TOKEN, 'Add a Colab secret named DAGSHUB_TOKEN (with notebook access).'

# Storage split for speed + persistence:
#  * RAW zips stay CACHED on Drive (avoid multi-GB re-downloads).
#  * The converted YOLO dataset is cached as a SINGLE tarball on Drive
#    (DATASET_CACHE) and extracted to fast local /content each session --
#    a single big file avoids the Drive small-file FUSE hang, and skips
#    re-converting on every runtime.
#  * Training checkpoints DO live on Drive (RUNS_DIR) with save_period=5, so a
#    disconnect auto-resumes from last.pt on the next Run All (no manual steps).
#  * The final best.pt is also copied to Drive in the last cell.
RAW_DIR = f'{DRIVE_ROOT}/data/raw'                  # cached on Drive (survives recycles)
YOLO_DIR = '/content/data/yolo'                     # fast local disk (extracted from cache)
DATA_YAML = '/content/data/yolo/VisDrone-DET/data.yaml'  # written by convert (below)
DATASET_CACHE = f'{DRIVE_ROOT}/cache/visdrone_yolo.tar.gz'  # single-file dataset cache on Drive
RUNS_DIR = f'{DRIVE_ROOT}/runs/train'               # checkpoints on Drive -> auto-resume
WEIGHTS_DIR = f'{DRIVE_ROOT}/weights'               # final best.pt persisted to Drive
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(os.path.dirname(DATASET_CACHE), exist_ok=True)

EPOCHS = 50
IMGSZ = 640
BATCH = 16
EXPERIMENT = 'visdrone-yolov8m'

## Data preparation

Clone this repo to reuse the conversion code, then download + convert VisDrone **into the Drive folder** so it is cached across sessions. The download step is resumable; on a reconnect just re-run and it skips already-extracted archives.

In [ ]:
# --- Get the repo (resumable clone; safe to re-run after a reconnect) ---
import os
if not os.path.isdir('object-detection-tracking'):
    !git clone https://github.com/mohanemg07-web/object-detection-tracking.git
%cd object-detection-tracking
!pip -q install -r requirements.txt

# --- Dataset: tarball-cache on Drive, extracted to fast local /content ---
# IDEMPOTENT: if the single-file cache exists on Drive, extract it (seconds) and
# SKIP download+convert entirely. Otherwise download (raw zips cached on Drive),
# convert to /content, then tar the result to Drive so future runtimes are fast.
import os, tarfile

if os.path.exists(DATASET_CACHE):
    print('Dataset cache found on Drive -> extracting to /content (skip convert):', DATASET_CACHE)
    os.makedirs(YOLO_DIR, exist_ok=True)
    with tarfile.open(DATASET_CACHE, 'r:gz') as tar:
        tar.extractall('/content/data/yolo')
    print('Extracted. data.yaml ->', DATA_YAML)
else:
    print('No dataset cache -> download + convert (one-time), then cache to Drive.')
    # Point the data pipeline at the cached raw dir (Drive) + local yolo dir.
    import pathlib, yaml
    paths = yaml.safe_load(open('configs/paths.yaml'))
    paths['raw_dir'] = RAW_DIR          # cached zips on Drive
    paths['yolo_dir'] = YOLO_DIR        # converted dataset on local /content
    paths['data_yaml'] = DATA_YAML      # generated data.yaml on local /content
    pathlib.Path('configs/paths.colab.yaml').write_text(yaml.safe_dump(paths))
    print('raw  ->', paths['raw_dir'])
    print('yolo ->', paths['yolo_dir'])
    print('yaml ->', paths['data_yaml'])

    # Resumable: re-running skips already-extracted raw archives.
    !python -m src.data.download_visdrone --config configs/paths.colab.yaml
    !python -m src.data.convert_visdrone  --config configs/paths.colab.yaml --write-data-yaml
    !python -m src.data.validate_labels   --config configs/paths.colab.yaml

    # Cache the converted dataset as ONE tarball on Drive (no small-file FUSE hang).
    print('Creating dataset cache on Drive (one-time):', DATASET_CACHE)
    tmp_cache = '/content/visdrone_yolo.tar.gz'  # build locally, then move to Drive
    with tarfile.open(tmp_cache, 'w:gz') as tar:
        tar.add('/content/data/yolo/VisDrone-DET', arcname='VisDrone-DET')
    import shutil
    shutil.move(tmp_cache, DATASET_CACHE)
    print('Dataset cached. Future runtimes will extract this instead of re-converting.')

assert os.path.exists(DATA_YAML), f'data.yaml missing after dataset prep: {DATA_YAML}'

In [ ]:
# --- Train with MLflow logging to DagsHub ---
import mlflow
from pathlib import Path
from ultralytics import YOLO, settings

# Disable Ultralytics' built-in MLflow integration so it doesn't open a
# SECOND, competing run alongside our manual logging below.
settings.update({'mlflow': False})

if MLFLOW_TRACKING_URI:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(EXPERIMENT)

# Resume from the last checkpoint on Drive if a previous (interrupted) run
# exists; otherwise start fresh. NOTE: a hardcoded resume=True would crash
# the very first run ('Resume checkpoint not found'), so make it conditional.
last_ckpt = Path(RUNS_DIR) / 'yolov8m_visdrone' / 'weights' / 'last.pt'
RESUME = last_ckpt.exists()
if RESUME:
    print('Resuming from', last_ckpt)
    model = YOLO(str(last_ckpt))   # resume must load FROM the checkpoint
else:
    print('No checkpoint found - starting a fresh run')
    model = YOLO('yolov8m.pt')

def _log_epoch(trainer):
    if not MLFLOW_TRACKING_URI:
        return
    m = {k.replace('(', '').replace(')', ''): float(v) for k, v in trainer.metrics.items()}
    mlflow.log_metrics(m, step=trainer.epoch)

model.add_callback('on_fit_epoch_end', _log_epoch)

run = mlflow.start_run() if MLFLOW_TRACKING_URI else None
results = model.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=0, seed=42, cos_lr=True, close_mosaic=10,
    project=RUNS_DIR, name='yolov8m_visdrone',  # checkpoints persisted to Drive
    save_period=5,  # checkpoint every 5 epochs so a disconnect loses <=5 epochs
    resume=RESUME, exist_ok=True,  # reconnect resumes from last.pt instead of making _visdrone2/
)
print('Saved to:', results.save_dir)

In [ ]:
# --- Log final metrics + best.pt as artifacts, copy best.pt to Drive, close run ---
import shutil
from pathlib import Path

save_dir = Path(results.save_dir)
best = save_dir / 'weights' / 'best.pt'

if MLFLOW_TRACKING_URI:
    if model.metrics and model.metrics.results_dict:
        mlflow.log_metrics({k.replace('(', '').replace(')', ''): float(v)
                            for k, v in model.metrics.results_dict.items()
                            if isinstance(v, (int, float))})
    for p in ['confusion_matrix.png', 'results.png', 'PR_curve.png']:
        fp = save_dir / p
        if fp.exists():
            mlflow.log_artifact(str(fp), 'plots')
    if best.exists():
        mlflow.log_artifact(str(best), 'weights')
    mlflow.end_run()

# Persist best.pt to a stable Drive location so a disconnect never loses it.
if best.exists():
    dest = Path(WEIGHTS_DIR) / 'best.pt'
    shutil.copy2(best, dest)
    print('best.pt copied to Drive:', dest)
print('best.pt:', best, '| exists:', best.exists())

## After training

`best.pt` is on Drive at `MyDrive/object-detection-tracking/weights/best.pt`. Download it and place it at `weights/best.pt` in your local repo, then resume with **Phase 4 (ONNX export + INT8 quantization)**. Paste the real mAP@0.5 / mAP@0.5:0.95 into the README results table.